# Prototype Analysis on Sample

Dieses Notebook testet erste Analyseideen auf einem kleinen Sample der bereinigten Parking-Violations-Daten.

## Ziel

- processed Parquet-Daten aus HDFS laden
- kleines 1%-Sample erstellen
- Analysefragen testen
- häufigste Violation Codes untersuchen
- häufigste Vehicle Makes untersuchen
- zeitliche Muster nach Monat, Wochentag und Tageszeit prüfen
- beurteilen, welche Queries und Charts für die finale Analyse sinnvoll sind

Die finalen Analysen werden später in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Prototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/24 09:24:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned"

df = spark.read.parquet(processed_path)

df.groupBy("fiscal_year").count().orderBy("fiscal_year").show()

+-----------+--------+
|fiscal_year|   count|
+-----------+--------+
|       2023|21562121|
|       2024|11848421|
|       2025|16557021|
+-----------+--------+



In [3]:
sample_df = df.sample(fraction=0.01, seed=42)

sample_count = sample_df.count()
sample_count

500468

In [4]:
sample_df.groupBy("violation_code", "violation_description_official") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)


[Stage 7:=============================================>           (15 + 4) / 19]

+--------------+------------------------------+------+
|violation_code|violation_description_official|count |
+--------------+------------------------------+------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|168862|
|21            |NO PARKING-STREET CLEANING    |58523 |
|38            |FAIL TO DSPLY MUNI METER RECPT|35045 |
|14            |NO STANDING-DAY/TIME LIMITS   |25096 |
|5             |BUS LANE VIOLATION            |21416 |
|7             |FAILURE TO STOP AT RED LIGHT  |20898 |
|40            |FIRE HYDRANT                  |20164 |
|20            |NO PARKING-DAY/TIME LIMITS    |19271 |
|71            |INSP. STICKER-EXPIRED/MISSING |17641 |
|70            |REG. STICKER-EXPIRED/MISSING  |12001 |
|46            |DOUBLE PARKING                |10528 |
|37            |EXPIRED MUNI METER            |7933  |
|31            |NO STANDING-COMM METER ZONE   |7889  |
|19            |NO STANDING-BUS STOP          |7290  |
|74            |FRONT OR BACK PLATE MISSING   |7242  |
|69       

In [5]:
sample_df.groupBy("violation_code") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 10:===================================>                    (12 + 4) / 19]

+--------------+------+
|violation_code|count |
+--------------+------+
|36            |168862|
|21            |58523 |
|38            |35045 |
|14            |25096 |
|5             |21416 |
|7             |20898 |
|40            |20164 |
|20            |19271 |
|71            |17641 |
|70            |12001 |
|46            |10528 |
|37            |7933  |
|31            |7889  |
|19            |7290  |
|74            |7242  |
|69            |7050  |
|16            |6993  |
|12            |5393  |
|43            |5208  |
|15            |4895  |
+--------------+------+
only showing top 20 rows



In [6]:
sample_df.groupBy("vehicle_make") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

[Stage 13:=========================================>              (14 + 4) / 19]

+------------+-----+
|vehicle_make|count|
+------------+-----+
|HONDA       |59786|
|TOYOT       |58524|
|FORD        |46580|
|NISSA       |39143|
|CHEVR       |26716|
|ME/BE       |25581|
|BMW         |24904|
|JEEP        |23068|
|HYUND       |17202|
|LEXUS       |12422|
|FRUEH       |11159|
|ACURA       |11108|
|SUBAR       |11051|
|KIA         |10288|
|DODGE       |9808 |
|AUDI        |9591 |
|MAZDA       |9152 |
|VOLKS       |9131 |
|INFIN       |7420 |
|RAM         |7366 |
+------------+-----+
only showing top 20 rows



In [7]:
sample_df.groupBy("fiscal_year", "issue_month") \
    .count() \
    .orderBy("fiscal_year", "issue_month") \
    .show(50)

+-----------+-----------+-----+
|fiscal_year|issue_month|count|
+-----------+-----------+-----+
|       2023|          1|13452|
|       2023|          2|12919|
|       2023|          3|15138|
|       2023|          4|14201|
|       2023|          5|15167|
|       2023|          6|16775|
|       2023|          7|28393|
|       2023|          8|32836|
|       2023|          9|25282|
|       2023|         10|15205|
|       2023|         11|14627|
|       2023|         12|12628|
|       2024|          1|12142|
|       2024|          2|12567|
|       2024|          3|13330|
|       2024|          4|13038|
|       2024|          5|14550|
|       2024|          6|10792|
|       2024|          8|   17|
|       2024|          9| 2613|
|       2024|         10|14263|
|       2024|         11|13528|
|       2024|         12|11781|
|       2025|          1|12025|
|       2025|          2|11757|
|       2025|          3|14650|
|       2025|          4|14704|
|       2025|          5|14426|
|       

In [8]:
sample_df.groupBy("issue_weekday") \
    .count() \
    .orderBy("issue_weekday") \
    .show()

[Stage 19:===============================================>        (16 + 3) / 19]

+-------------+-----+
|issue_weekday|count|
+-------------+-----+
|            1|48356|
|            2|72537|
|            3|81314|
|            4|75005|
|            5|80810|
|            6|79547|
|            7|62899|
+-------------+-----+



In [9]:
sample_df.filter(col("violation_hour").isNotNull()) \
    .groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(24)

[Stage 22:===============================================>        (16 + 3) / 19]

+--------------+-----+
|violation_hour|count|
+--------------+-----+
|             0| 6109|
|             1| 7435|
|             2| 5850|
|             3| 4771|
|             4| 4462|
|             5| 7016|
|             6|15388|
|             7|27806|
|             8|42612|
|             9|43849|
|            10|36116|
|            11|43734|
|            12|40996|
|            13|39166|
|            14|35285|
|            15|29970|
|            16|23901|
|            17|20683|
|            18|15323|
|            19|11356|
|            20|11092|
|            21| 9985|
|            22| 8645|
|            23| 7732|
+--------------+-----+



## Erkenntnisse aus dem Prototyping

Das 1%-Sample enthält 500'468 Zeilen und ist damit gross genug, um Analyseideen zu testen.

Getestete Analysefragen:

1. **Welche Violation Codes kommen am häufigsten vor?**  
   `violation_code = 36` (PHTO SCHOOL ZN SPEED VIOLATION) ist im Sample mit Abstand am häufigsten. Für die finale Analyse wird `violation_description_officia` verwendet, da diese eine eindeutige offizielle Beschreibung pro Code liefert.

2. **Welche Vehicle Makes erhalten am häufigsten Parking Violations?**  
   Im Sample sind `HONDA`, `TOYOT`, `FORD` und `NISSA` besonders häufig.

3. **Gibt es zeitliche Muster nach Monat?**  
   Besonders FY2023 zeigt auffällige Werte in den Monaten Juli, August und September. Bei FY2024 fehlen diese Monate fast vollständig — das ist eine direkte Folge der Deduplizierung auf `summons_number`: Violations aus Juli–September 2023 erschienen sowohl in FY2023 als auch in FY2024 und wurden als Duplikate entfernt.

4. **Gibt es zeitliche Muster nach Wochentag?**  
   Sonntag (Wochentag 1) weist deutlich weniger Verstösse auf als die Werktage.

5. **Gibt es zeitliche Muster nach Tageszeit?**  
   Im Sample zeigen sich hohe Werte insbesondere zwischen ca. 08:00 und 14:00 Uhr. Für diese Analyse werden nur Datensätze mit `violation_hour IS NOT NULL` verwendet.

Die getesteten Analysen werden im nächsten Schritt in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [10]:
spark.stop()